# HealthBandhu V

## AI-powered multiclass disease screening and clinical decision support

Author: Joy Ghatak

This notebook trains every model used by the HealthBandhu V web app (`App.py`) and saves them to the `models/` folder.

**Run in Google Colab:** Runtime > Run all. The first cell asks you to upload `HEALTHBANDHU V.zip` (skip if the extracted folder is already in `/content`). A GPU runtime is optional; training also works on CPU.

**Run locally (Jupyter, Conda):** open this notebook from the project's `notebooks/` folder and run all cells.

The dataset (about 190 MB) downloads automatically to `data/medical_dataset.csv` on the first run. Every problem fixed from the previous notebook is listed in `docs/FIXES.md`.

# HealthBandhu architecture

HealthBandhu combines several independent layers so that no single model decides alone:

- Emergency detection layer (red-flag scoring, runs first)
- Rule-based clinical layer (key-symptom profiles for every disease)
- Machine learning layer (Bernoulli Naive Bayes, plus an Extra Trees baseline for comparison)
- Deep learning layer (multilayer neural network)
- Ensemble intelligence (weight tuned on validation data)
- Explainable AI (occlusion attribution)
- Confidence layer (checked against test accuracy)
- Clinical assistant and automated PDF report

All inference code lives in the `healthbandhu` package, shared with the web app, so the notebook and the app always behave identically.

## Phase roadmap

| Phase | Content |
|---|---|
| Setup | Locate project, install packages, seeds |
| 0 | Dataset intelligence |
| 1 | Exploratory data analysis |
| 2 | Data preprocessing |
| 3 | Emergency detection layer |
| 4 | Rule-based clinical layer |
| 5 | Machine learning layer |
| 6 | Deep learning layer |
| 7 | Model comparison |
| 8 | Ensemble intelligence |
| 9 | Explainable AI |
| 10 | Confidence layer |
| 11 | Clinical assistant |
| 12 | Automated report generator |
| 13 | Evaluation and model saving |
| 14 | Universal prediction pipeline |
| 15 | Packaging and launch |

# Setup

In [ ]:
# =====================================================
# SETUP: LOCATE PROJECT AND INSTALL PACKAGES
# =====================================================

import gc
import importlib.util
import json
import os
import random
import re
import subprocess
import sys
import time
import zipfile
from datetime import datetime
from pathlib import Path

PROJECT_NAME = "HEALTHBANDHU V"
IN_COLAB = "google.colab" in sys.modules


def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content") / PROJECT_NAME,
        Path("/content/drive/MyDrive") / PROJECT_NAME,
    ]
    for candidate in candidates:
        if (candidate / "healthbandhu" / "__init__.py").exists() and (candidate / "App.py").exists():
            return candidate.resolve()
    return None


PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None and IN_COLAB:
    from google.colab import files

    zips = sorted(Path("/content").glob("HEALTHBANDHU*.zip"))
    if not zips:
        print(f"Upload '{PROJECT_NAME}.zip' (the project zip file).")
        uploaded = files.upload()
        zips = [Path("/content") / name for name in uploaded if name.lower().endswith(".zip")]
    if not zips:
        raise FileNotFoundError("No project zip was uploaded.")
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall("/content")
    PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "HealthBandhu project not found. Open this notebook from the project's 'notebooks' "
        f"folder, or upload '{PROJECT_NAME}.zip' when running in Colab."
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def ensure_package(pip_name, import_name):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)


for pip_name, import_name in [("gdown", "gdown"), ("reportlab", "reportlab"),
                              ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib")]:
    ensure_package(pip_name, import_name)

print("Project root :", PROJECT_ROOT)
print("Google Colab :", IN_COLAB)

In [ ]:
# =====================================================
# IMPORTS, CONFIGURATION AND REPRODUCIBILITY
# =====================================================

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from healthbandhu import config
from healthbandhu.clinical import assess_confidence
from healthbandhu.emergency import assess_emergency, emergency_scores, emergency_symptoms_in
from healthbandhu.explain import global_occlusion_importance
from healthbandhu.models import (HealthBandhuPredictor, ModelBundle, load_bundle, naive_bayes_proba,
                                 top_k_indices)
from healthbandhu.report import build_pdf_report
from healthbandhu.rules import build_disease_profiles, key_symptom_mask, rule_scores, top_rule_matches

SEED = config.RANDOM_SEED
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

for folder in (config.DATA_DIR, config.MODELS_DIR, config.REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

TARGET = config.TARGET_COLUMN
TRAIN_EXTRA_TREES_BASELINE = True   # comparison only, never saved (the old model file was 880 MB)
EPOCHS = 30                         # early stopping usually ends training sooner
BATCH_SIZE = 1024

VERSIONS = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "tensorflow": tf.__version__,
    "keras": tf.keras.__version__,
}
print(json.dumps(VERSIONS, indent=2))
print("GPU:", tf.config.list_physical_devices("GPU") or "none (CPU training)")

# Phase 0: Dataset intelligence

Loads the symptom dataset with compact `uint8` columns (the old notebook used `int64`, about 726 MB in memory), then checks the target, class counts, value ranges and duplicated columns.

In [ ]:
# =====================================================
# UNIVERSAL DATASET LOADER
# =====================================================

if not config.DATASET_PATH.exists():
    import gdown

    print("Downloading dataset ...")
    gdown.download(f"https://drive.google.com/uc?id={config.DATASET_GDRIVE_FILE_ID}",
                   str(config.DATASET_PATH), quiet=False)

header = pd.read_csv(config.DATASET_PATH, nrows=0).columns.tolist()
dtypes = {col: "uint8" for col in header if col != TARGET}
dtypes[TARGET] = str
try:
    df_raw = pd.read_csv(config.DATASET_PATH, dtype=dtypes)
except (ValueError, TypeError):
    df_raw = pd.read_csv(config.DATASET_PATH)
    feature_cols = [c for c in df_raw.columns if c != TARGET]
    df_raw[feature_cols] = df_raw[feature_cols].fillna(0).clip(0, 1).astype("uint8")
    df_raw[TARGET] = df_raw[TARGET].astype(str)

df_raw[TARGET] = df_raw[TARGET].str.strip()

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)
print("Shape          :", df_raw.shape)
print("Memory (MB)    :", round(df_raw.memory_usage(deep=True).sum() / 1024**2, 1))
display(df_raw.head())

In [ ]:
# =====================================================
# DATASET INTELLIGENCE REPORT
# =====================================================

raw_symptom_cols = [c for c in df_raw.columns if c != TARGET]
disease_counts = df_raw[TARGET].value_counts()

# pandas renames a repeated header "x" to "x.1"; these are the same symptom twice
DUPLICATE_COLUMNS = {}
for col in raw_symptom_cols:
    base = re.sub(r"\.\d+$", "", col)
    if base != col and base in raw_symptom_cols:
        DUPLICATE_COLUMNS[col] = base

non_binary = int((df_raw[raw_symptom_cols] > 1).to_numpy().sum())
missing_values = int(df_raw.isna().sum().sum())

print("=" * 60)
print("DATASET INTELLIGENCE REPORT")
print("=" * 60)
print(f"Rows                     : {len(df_raw):,}")
print(f"Symptom columns          : {len(raw_symptom_cols)}")
print(f"Disease classes (raw)    : {df_raw[TARGET].nunique()}")
print(f"Missing values           : {missing_values}")
print(f"Values other than 0 or 1 : {non_binary}")
print(f"Duplicated symptom cols  : {DUPLICATE_COLUMNS or 'none'}")
print(f"Largest class            : {disease_counts.max()} records")
print(f"Smallest class           : {disease_counts.min()} records")
print(f"Imbalance ratio          : {disease_counts.max() / disease_counts.min():.0f} : 1")

print("\nMost frequent diseases")
display(disease_counts.head(10).to_frame("records"))
print("Least frequent diseases")
display(disease_counts.tail(10).to_frame("records"))

In [ ]:
# =====================================================
# CLASS DISTRIBUTION
# =====================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].plot(np.arange(len(disease_counts)), disease_counts.values, color="#2F3E8F")
axes[0].set_yscale("log")
axes[0].set_title("Records per disease, sorted (log scale)")
axes[0].set_xlabel("Disease rank")
axes[0].set_ylabel("Records")

bins = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000]
axes[1].hist(disease_counts.values, bins=bins, color="#2F3E8F", edgecolor="white")
axes[1].set_xscale("log")
axes[1].set_title("How many diseases have N records")
axes[1].set_xlabel("Records per disease (log scale)")
axes[1].set_ylabel("Number of diseases")

plt.tight_layout()
plt.savefig(config.REPORTS_DIR / "class_distribution.png", dpi=150)
plt.show()

for limit in (2, 5, 10, 20):
    print(f"Diseases with fewer than {limit:>2} records: {(disease_counts < limit).sum()}")

# Phase 1: Exploratory data analysis

Symptom frequency, sparsity, symptoms per patient and duplicate records. Duplicates matter: identical symptom vectors with different diagnoses set a ceiling on the accuracy any model can reach.

In [ ]:
# =====================================================
# SYMPTOM ACTIVATION AND SPARSITY
# =====================================================

activation_counts = df_raw[raw_symptom_cols].sum().sort_values(ascending=False)
symptoms_per_patient = df_raw[raw_symptom_cols].sum(axis=1)

print(f"Average sparsity             : {100 * (1 - activation_counts.sum() / df_raw[raw_symptom_cols].size):.2f}% zeros")
print(f"Symptoms never reported      : {(activation_counts == 0).sum()}")
print(f"Symptoms reported < 10 times : {(activation_counts < 10).sum()}")
print(f"Symptoms per patient         : mean {symptoms_per_patient.mean():.2f}, "
      f"median {symptoms_per_patient.median():.0f}, max {symptoms_per_patient.max()}")
print(f"Patients with no symptoms    : {(symptoms_per_patient == 0).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
top = activation_counts.head(25)[::-1]
axes[0].barh(top.index, top.values, color="#2F3E8F")
axes[0].set_title("25 most frequent symptoms")
axes[0].set_xlabel("Records")

axes[1].hist(symptoms_per_patient, bins=range(0, int(symptoms_per_patient.max()) + 2), color="#2F3E8F",
             edgecolor="white")
axes[1].set_title("Symptoms reported per patient")
axes[1].set_xlabel("Number of symptoms")
axes[1].set_ylabel("Patients")

plt.tight_layout()
plt.savefig(config.REPORTS_DIR / "symptom_analysis.png", dpi=150)
plt.show()

In [ ]:
# =====================================================
# DUPLICATE AND CONFLICTING RECORDS
# =====================================================

feature_hash = pd.util.hash_pandas_object(df_raw[raw_symptom_cols], index=False)
record_hash = pd.util.hash_pandas_object(df_raw, index=False)

exact_duplicate_rows = int(record_hash.duplicated().sum())
labels_per_vector = df_raw.groupby(feature_hash.values)[TARGET].nunique()
conflicting_vectors = int((labels_per_vector > 1).sum())
rows_in_conflict = int(feature_hash.map(labels_per_vector).gt(1).sum())

print(f"Exact duplicate records (same symptoms and disease) : {exact_duplicate_rows:,}")
print(f"Symptom patterns linked to more than one disease    : {conflicting_vectors:,}")
print(f"Records sharing a pattern with another disease      : {rows_in_conflict:,} "
      f"({100 * rows_in_conflict / len(df_raw):.1f}%)")
print("\nRecords that share an identical symptom pattern with a different disease cannot all be "
      "classified correctly, so 100% accuracy is impossible on this dataset.")

del feature_hash, record_hash, labels_per_vector
gc.collect();

# Phase 2: Data preprocessing

1. Merge duplicated symptom columns (for example `regurgitation` and `regurgitation.1`) with a logical OR.
2. Remove diseases with fewer than `MIN_SAMPLES_PER_CLASS` records (they cannot be stratified). This is done once; the old notebook repeated it and fitted the label encoder twice.
3. Split into train, validation and test sets. The test set is only used for final evaluation; weights, smoothing and temperature are tuned on validation data.
4. Remove symptoms that never occur in the training split. A network input that is always zero keeps its random initial weights, so selecting that symptom in the app would add noise.

In [ ]:
# =====================================================
# CLEAN, ENCODE AND SPLIT
# =====================================================

df = df_raw.copy()

for duplicate, base in DUPLICATE_COLUMNS.items():
    df[base] = (df[base] | df[duplicate]).astype("uint8")
df = df.drop(columns=list(DUPLICATE_COLUMNS))

class_sizes = df[TARGET].value_counts()
REMOVED_CLASSES = sorted(class_sizes[class_sizes < config.MIN_SAMPLES_PER_CLASS].index.tolist())
df = df[~df[TARGET].isin(REMOVED_CLASSES)].reset_index(drop=True)

label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(df[TARGET]).astype(np.int32)
DISEASES = label_encoder.classes_.tolist()
N_CLASSES = len(DISEASES)

candidate_symptoms = [c for c in df.columns if c != TARGET]
X_all = df[candidate_symptoms].to_numpy(dtype=np.uint8)

all_idx = np.arange(len(df))
trainval_idx, test_idx = train_test_split(all_idx, test_size=config.TEST_SIZE, random_state=SEED, stratify=y_all)
# Not stratified: after the first split some classes have a single record left.
train_idx, val_idx = train_test_split(trainval_idx, test_size=config.VAL_SIZE, random_state=SEED)

# Every class must appear in training; move any validation-only records back.
missing_in_train = np.setdiff1d(np.arange(N_CLASSES), np.unique(y_all[train_idx]))
if missing_in_train.size:
    moved = val_idx[np.isin(y_all[val_idx], missing_in_train)]
    train_idx = np.concatenate([train_idx, moved])
    val_idx = np.setdiff1d(val_idx, moved)
    print(f"Moved {moved.size} validation records to training so every class is learnable.")

keep = X_all[train_idx].sum(axis=0) > 0
DROPPED_SYMPTOMS = [s for s, k in zip(candidate_symptoms, keep) if not k]
SYMPTOMS = [s for s, k in zip(candidate_symptoms, keep) if k]
N_FEATURES = len(SYMPTOMS)
X_all = X_all[:, keep]

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val, y_val = X_all[val_idx], y_all[val_idx]
X_test, y_test = X_all[test_idx], y_all[test_idx]
class_train_counts = np.bincount(y_train, minlength=N_CLASSES)

print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"Merged duplicate columns : {DUPLICATE_COLUMNS or 'none'}")
print(f"Removed rare diseases    : {len(REMOVED_CLASSES)}")
print(f"Removed unused symptoms  : {len(DROPPED_SYMPTOMS)} {DROPPED_SYMPTOMS[:10]}")
print(f"Records used             : {len(df):,}")
print(f"Symptoms (features)      : {N_FEATURES}")
print(f"Disease classes          : {N_CLASSES}")
print(f"Train / val / test       : {len(y_train):,} / {len(y_val):,} / {len(y_test):,}")
print(f"Classes in train / test  : {np.unique(y_train).size} / {np.unique(y_test).size}")

del df, X_all
gc.collect();

# Phase 3: Emergency detection layer

Red-flag symptoms are scored before any disease prediction. Weights and thresholds live in `healthbandhu/config.py` (edit them there, not here).

Priority: emergency screening, then disease prediction, then the clinical report.

In [ ]:
# =====================================================
# EMERGENCY DETECTION SYSTEM
# =====================================================

train_frequency = X_train.mean(axis=0)
symptom_position = {s: i for i, s in enumerate(SYMPTOMS)}

emergency_table = pd.DataFrame([
    {"symptom": s, "weight": w, "in model": s in symptom_position,
     "training frequency %": round(100 * train_frequency[symptom_position[s]], 3) if s in symptom_position else None}
    for s, w in config.EMERGENCY_SYMPTOMS.items()
])
display(emergency_table)
print("Levels: " + ", ".join(f"{level} >= {t}" for level, t in config.EMERGENCY_LEVELS) + f", otherwise {config.LOW_RISK}")

train_emergency = emergency_scores(X_train, SYMPTOMS)
level_names = [level for level, _ in config.EMERGENCY_LEVELS] + [config.LOW_RISK]
conditions = [train_emergency >= t for _, t in config.EMERGENCY_LEVELS]
train_levels = np.select(conditions, level_names[:-1], default=config.LOW_RISK)

print(f"\nMaximum score : {train_emergency.max()}")
print(f"Average score : {train_emergency.mean():.2f}")
display(pd.Series(train_levels).value_counts().reindex(level_names).fillna(0).astype(int)
        .to_frame("training records").assign(percent=lambda t: (100 * t["training records"] / len(train_levels)).round(2)))

In [ ]:
# =====================================================
# EMERGENCY ASSESSMENT FOR ONE PATIENT
# =====================================================

urgent_rows = np.flatnonzero(emergency_scores(X_test, SYMPTOMS) >= config.EMERGENCY_LEVELS[0][1])
example_row = int(urgent_rows[0]) if urgent_rows.size else 0
example_symptoms = [SYMPTOMS[i] for i in np.flatnonzero(X_test[example_row])]

assessment = assess_emergency(example_symptoms)
print("Symptoms :", example_symptoms)
print("Score    :", assessment.score)
print("Level    :", assessment.level)
print("Triggers :", assessment.triggers)
print("Action   :", assessment.action)

# Phase 4: Rule-based clinical layer

Builds a symptom profile for **every** disease from the training split only (the old notebook covered 20 diseases and used the full dataset, including test records). Symptoms present in at least 25% of a disease's patients become its key symptoms.

Rule score = 0.7 x F1 overlap with the key symptoms + 0.3 x average profile frequency of the reported symptoms.

In [ ]:
# =====================================================
# DISEASE SYMPTOM PROFILES
# =====================================================

profiles, _ = build_disease_profiles(X_train, y_train, N_CLASSES)
key_mask = key_symptom_mask(profiles)
key_counts = key_mask.sum(axis=1)

print(f"Profiles built       : {profiles.shape[0]} diseases x {profiles.shape[1]} symptoms")
print(f"Key symptoms/disease : mean {key_counts.mean():.1f}, min {key_counts.min()}, max {key_counts.max()}")

for d in np.argsort(-class_train_counts)[:5]:
    key = np.flatnonzero(key_mask[d])
    key = key[np.argsort(-profiles[d, key])]
    print(f"\n{DISEASES[d]} ({class_train_counts[d]} training records)")
    print("   " + ", ".join(f"{SYMPTOMS[i]} {profiles[d, i]:.0%}" for i in key))

In [ ]:
# =====================================================
# METRICS HELPER AND RULE ENGINE EVALUATION
# =====================================================

RESULTS = {}


def evaluate(name, scores, y_true, deployed=False):
    """Top-k accuracy and F1 for a score matrix (rows = patients, columns = diseases)."""
    top5 = top_k_indices(scores, 5)
    pred = top5[:, 0]
    RESULTS[name] = {
        "top1": float((pred == y_true).mean()),
        "top3": float((top5[:, :3] == y_true[:, None]).any(axis=1).mean()),
        "top5": float((top5 == y_true[:, None]).any(axis=1).mean()),
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, pred, average="weighted", zero_division=0)),
        "deployed": bool(deployed),
    }
    r = RESULTS[name]
    print(f"{name:<24} top-1 {r['top1']:.4f} | top-3 {r['top3']:.4f} | top-5 {r['top5']:.4f} | "
          f"macro F1 {r['macro_f1']:.4f}")
    return r


def negative_log_likelihood(proba, y_true):
    return float(-np.mean(np.log(np.clip(proba[np.arange(len(y_true)), y_true], 1e-12, 1.0))))


rule_test_scores = rule_scores(X_test, profiles, key_mask)
evaluate("Rule-based", rule_test_scores, y_test, deployed=True)
del rule_test_scores

patient = X_test[example_row]
print("\nRule matches for the example patient (true disease:", DISEASES[y_test[example_row]], ")")
for m in top_rule_matches(patient, SYMPTOMS, DISEASES, profiles, key_mask):
    print(f"  {m.disease:<40} score {m.score:.2f} | matched {len(m.matched)} of {m.key_count}: {m.matched}")

# Phase 5: Machine learning layer

**Bernoulli Naive Bayes** is the classical model used in the app. It suits binary present/absent symptoms, trains in seconds and its parameters are two small arrays, saved as `.npz` so the app needs no pickle and no scikit-learn.

- Smoothing `alpha` is chosen by validation top-1 accuracy.
- A temperature is chosen by validation log loss, because Naive Bayes probabilities are extremely over-confident.
- The numpy implementation used by the app is checked against scikit-learn.

**Extra Trees** is kept as a baseline for comparison with the original design, but it is not saved: with 754 classes every tree node stores 754 values, which made the old model file 880 MB.

In [ ]:
# =====================================================
# BERNOULLI NAIVE BAYES: TUNE ON VALIDATION DATA
# =====================================================

from sklearn.naive_bayes import BernoulliNB

assert (class_train_counts > 0).all(), "Every class needs at least one training record"
order = np.argsort(y_train, kind="stable")
starts = np.searchsorted(y_train[order], np.arange(N_CLASSES))
feature_count = np.add.reduceat(X_train[order].astype(np.int32), starts, axis=0).astype(np.float64)
class_count = class_train_counts.astype(np.float64)
class_log_prior = np.log(class_count) - np.log(class_count.sum())


def nb_feature_log_prob(alpha):
    return np.log(feature_count + alpha) - np.log(class_count[:, None] + 2.0 * alpha)


alpha_rows = []
for alpha in [0.01, 0.03, 0.1, 0.3, 1.0]:
    val_proba = naive_bayes_proba(X_val, nb_feature_log_prob(alpha), class_log_prior)
    alpha_rows.append({"alpha": alpha, "val_top1": float((val_proba.argmax(axis=1) == y_val).mean())})
alpha_table = pd.DataFrame(alpha_rows)
display(alpha_table)
NB_ALPHA = float(alpha_table.sort_values(["val_top1", "alpha"], ascending=[False, False]).iloc[0]["alpha"])
nb_flp = nb_feature_log_prob(NB_ALPHA)

temp_rows = []
for temperature in [1, 2, 3, 5, 8, 12, 20, 30, 50]:
    val_proba = naive_bayes_proba(X_val, nb_flp, class_log_prior, temperature)
    temp_rows.append({"temperature": temperature, "val_log_loss": negative_log_likelihood(val_proba, y_val)})
temp_table = pd.DataFrame(temp_rows)
display(temp_table)
NB_TEMPERATURE = float(temp_table.sort_values("val_log_loss").iloc[0]["temperature"])
print(f"Selected alpha = {NB_ALPHA}, temperature = {NB_TEMPERATURE}")

In [ ]:
# =====================================================
# VERIFY THE APP'S NUMPY NAIVE BAYES AGAINST SCIKIT-LEARN
# =====================================================

sk_nb = BernoulliNB(alpha=NB_ALPHA).fit(X_train, y_train)
check_rows = X_val[:2000]
ours = naive_bayes_proba(check_rows, nb_flp, class_log_prior, temperature=1.0)
theirs = sk_nb.predict_proba(check_rows)
max_difference = float(np.abs(ours - theirs).max())

print(f"Max probability difference vs scikit-learn : {max_difference:.2e}")
print(f"Same predicted class                       : {np.mean(ours.argmax(1) == theirs.argmax(1)):.4%}")
assert np.array_equal(sk_nb.classes_, np.arange(N_CLASSES))
assert max_difference < 1e-4, "Numpy Naive Bayes does not match scikit-learn"

del sk_nb, ours, theirs
gc.collect()

nb_test = naive_bayes_proba(X_test, nb_flp, class_log_prior, NB_TEMPERATURE)
evaluate("Bernoulli Naive Bayes", nb_test, y_test, deployed=True);

In [ ]:
# =====================================================
# EXTRA TREES BASELINE (COMPARISON ONLY, NOT SAVED)
# =====================================================

if TRAIN_EXTRA_TREES_BASELINE:
    from sklearn.ensemble import ExtraTreesClassifier

    start = time.time()
    et_model = ExtraTreesClassifier(n_estimators=100, max_depth=20, n_jobs=-1,
                                    random_state=SEED, class_weight="balanced")
    et_model.fit(X_train, y_train)
    print(f"Training time: {time.time() - start:.1f} s")

    et_test = et_model.predict_proba(X_test).astype(np.float32)
    evaluate("Extra Trees (baseline)", et_test, y_test, deployed=False)

    nodes = sum(t.tree_.node_count for t in et_model.estimators_)
    print(f"Estimated model size if saved: {nodes * (N_CLASSES * 8 + 64) / 1024**2:,.0f} MB "
          f"({nodes:,} nodes x {N_CLASSES} class values each)")

    importance = (pd.DataFrame({"symptom": SYMPTOMS, "importance": et_model.feature_importances_})
                  .sort_values("importance", ascending=False).head(15))
    display(importance)

    del et_model, et_test
    gc.collect()
else:
    print("Extra Trees baseline skipped.")

# Phase 6: Deep learning layer

Same architecture as before (512-256-128 with batch normalisation and dropout), with these fixes:

- An explicit `Input` layer sized from the data, instead of a hard-coded `input_shape=(377,)` that triggered a Keras warning.
- A proper validation set, early stopping on validation loss and learning-rate reduction.
- Top-5 accuracy tracked during training.

In [ ]:
# =====================================================
# DNN MODEL
# =====================================================

tf.keras.utils.set_random_seed(SEED)

dnn_model = tf.keras.Sequential([
    tf.keras.Input(shape=(N_FEATURES,), name="symptoms"),
    tf.keras.layers.Dense(512, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(N_CLASSES, activation="softmax", name="disease_probabilities"),
], name="healthbandhu_dnn")

dnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")],
)
dnn_model.summary()

In [ ]:
# =====================================================
# DNN TRAINING
# =====================================================

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1),
]

start = time.time()
history = dnn_model.fit(
    X_train.astype("float32"), y_train,
    validation_data=(X_val.astype("float32"), y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=2,
)
print(f"Training time: {time.time() - start:.1f} s, epochs run: {len(history.history['loss'])}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric, title in [(axes[0], "loss", "Loss"), (axes[1], "accuracy", "Accuracy")]:
    ax.plot(history.history[metric], label="train", color="#2F3E8F")
    ax.plot(history.history[f"val_{metric}"], label="validation", color="#C2410C")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
plt.tight_layout()
plt.savefig(config.REPORTS_DIR / "dnn_training_curves.png", dpi=150)
plt.show()

In [ ]:
# =====================================================
# DNN EVALUATION
# =====================================================

dnn_test = dnn_model.predict(X_test.astype("float32"), batch_size=4096, verbose=0)
evaluate("Deep neural network", dnn_test, y_test, deployed=True);

# Phase 7: Model comparison

All metrics below are computed on the same held-out test set. Nothing is typed in by hand (the old notebook saved hard-coded metrics that did not match its own results).

In [ ]:
# =====================================================
# MODEL COMPARISON (INDIVIDUAL MODELS)
# =====================================================

comparison = pd.DataFrame(RESULTS).T[["top1", "top3", "top5", "macro_f1", "weighted_f1", "deployed"]]
display(comparison.style.format({c: "{:.4f}" for c in ["top1", "top3", "top5", "macro_f1", "weighted_f1"]}))

# Phase 8: Ensemble intelligence

The old notebook defined `DNN_WEIGHT = 0.80` and `ET_WEIGHT = 0.20` but never used them. Here the ensemble is real:

probability = w x DNN + (1 - w) x Naive Bayes

`w` is chosen on the validation set (highest top-1 accuracy, ties broken by lower log loss) and then evaluated once on the test set.

In [ ]:
# =====================================================
# ENSEMBLE WEIGHT SEARCH (VALIDATION SET)
# =====================================================

dnn_val = dnn_model.predict(X_val.astype("float32"), batch_size=4096, verbose=0)
nb_val = naive_bayes_proba(X_val, nb_flp, class_log_prior, NB_TEMPERATURE)

grid = []
for w in np.round(np.arange(0.0, 1.0001, 0.05), 2):
    blended = w * dnn_val + (1.0 - w) * nb_val
    grid.append({"dnn_weight": float(w),
                 "val_top1": float((blended.argmax(axis=1) == y_val).mean()),
                 "val_log_loss": negative_log_likelihood(blended, y_val)})
grid = pd.DataFrame(grid)
best = grid.sort_values(["val_top1", "val_log_loss"], ascending=[False, True]).iloc[0]
DNN_WEIGHT = float(best["dnn_weight"])

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(grid["dnn_weight"], grid["val_top1"], color="#2F3E8F", marker="o", label="validation top-1")
ax1.set_xlabel("DNN weight (Naive Bayes weight = 1 - w)")
ax1.set_ylabel("Validation top-1 accuracy", color="#2F3E8F")
ax2 = ax1.twinx()
ax2.plot(grid["dnn_weight"], grid["val_log_loss"], color="#C2410C", marker=".", label="validation log loss")
ax2.set_ylabel("Validation log loss", color="#C2410C")
ax1.axvline(DNN_WEIGHT, color="grey", linestyle="--")
plt.title(f"Ensemble weight search (selected w = {DNN_WEIGHT:.2f})")
plt.tight_layout()
plt.savefig(config.REPORTS_DIR / "ensemble_weight_search.png", dpi=150)
plt.show()

print(f"Selected DNN weight: {DNN_WEIGHT:.2f}  |  Naive Bayes weight: {1 - DNN_WEIGHT:.2f}")

In [ ]:
# =====================================================
# ENSEMBLE EVALUATION AND FINAL COMPARISON
# =====================================================

ensemble_test = DNN_WEIGHT * dnn_test + (1.0 - DNN_WEIGHT) * nb_test
evaluate("Ensemble", ensemble_test, y_test, deployed=True)

comparison = pd.DataFrame(RESULTS).T[["top1", "top3", "top5", "macro_f1", "weighted_f1", "deployed"]]
comparison.to_csv(config.REPORTS_DIR / "model_comparison.csv")
display(comparison.style.format({c: "{:.4f}" for c in ["top1", "top3", "top5", "macro_f1", "weighted_f1"]}))

ax = comparison[["top1", "top3", "top5"]].astype(float).plot(kind="barh", figsize=(10, 5),
                                                             color=["#2F3E8F", "#6F7BC2", "#B9C0E4"])
ax.set_xlim(0, 1)
ax.set_xlabel("Accuracy on the test set")
ax.set_title("Model performance comparison")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(config.REPORTS_DIR / "model_comparison.png", dpi=150)
plt.show()

# Phase 9: Explainable AI

The old "explanation" only listed the symptoms the patient reported. HealthBandhu now uses **occlusion attribution**: each reported symptom is removed in turn and the change in the predicted disease's log-probability is measured. Each symptom's **influence** is its signed share of that total, which stays informative even when the probability is close to 100%. The same code runs in the web app.

From here on, the notebook uses `HealthBandhuPredictor`, the exact inference class used by `App.py`.

In [ ]:
# =====================================================
# BUILD THE SHARED PREDICTOR FROM THE TRAINED MODELS
# =====================================================

bundle = ModelBundle(
    symptoms=SYMPTOMS,
    diseases=DISEASES,
    dnn=dnn_model,
    nb_feature_log_prob=nb_flp,
    nb_class_log_prior=class_log_prior,
    nb_temperature=NB_TEMPERATURE,
    profiles=profiles,
    dnn_weight=DNN_WEIGHT,
    metadata={"class_train_counts": class_train_counts.tolist()},
)
predictor = HealthBandhuPredictor(bundle)

check = X_test[:256].astype("float32")
assert np.allclose(predictor.ensemble_proba(check), ensemble_test[:256], atol=1e-5)
print("Predictor reproduces the evaluated ensemble:", predictor.models_used)

In [ ]:
# =====================================================
# SAMPLE PATIENT EXPLANATION
# =====================================================

symptom_counts = X_test.sum(axis=1)
sample_index = int(np.flatnonzero(symptom_counts >= 5)[0])
sample_symptoms = [SYMPTOMS[i] for i in np.flatnonzero(X_test[sample_index])]
result = predictor.diagnose(sample_symptoms)

print("=" * 60)
print("AI EXPLANATION")
print("=" * 60)
print("Reported symptoms :", ", ".join(result.symptoms))
print("True disease      :", DISEASES[y_test[sample_index]])
print()
display(pd.DataFrame([{
    "rank": c.rank, "condition": c.name, "ensemble": round(c.probability, 4),
    "dnn": round(c.dnn_probability, 4), "naive bayes": round(c.nb_probability, 4),
    "rule score": round(c.rule_score, 3), "training records": c.train_count,
} for c in result.candidates]))

contrib = pd.DataFrame([c.to_dict() for c in result.contributions])
contrib["influence (%)"] = (contrib["influence"] * 100).round(1)
contrib["probability drop (points)"] = (contrib["impact"] * 100).round(3)
display(contrib[["symptom", "influence (%)", "probability drop (points)"]])

plot_df = contrib.iloc[::-1]
plt.figure(figsize=(9, 0.45 * len(plot_df) + 1.5))
plt.barh(plot_df["symptom"], plot_df["influence (%)"],
         color=["#2F3E8F" if v >= 0 else "#A3A9BC" for v in plot_df["influence (%)"]])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Share of the evidence for the predicted condition (%)")
plt.title(f"Why '{result.top.name}' ranked first")
plt.tight_layout()
plt.savefig(config.REPORTS_DIR / "xai_sample_patient.png", dpi=150)
plt.show()

print("Often seen with this condition but not reported:", result.unreported_typical)

In [ ]:
# =====================================================
# GLOBAL SYMPTOM IMPORTANCE (OCCLUSION, 300 TEST PATIENTS)
# =====================================================

sample_rng = np.random.default_rng(SEED)
sample_rows = sample_rng.choice(len(X_test), size=min(300, len(X_test)), replace=False)
global_importance = global_occlusion_importance(predictor.ensemble_proba, X_test[sample_rows], SYMPTOMS)

importance_df = pd.DataFrame(global_importance, columns=["symptom", "mean influence", "times reported"])
importance_df = importance_df[importance_df["times reported"] >= 5].head(20)
importance_df["mean influence (%)"] = (importance_df["mean influence"] * 100).round(1)
importance_df.to_csv(config.REPORTS_DIR / "global_symptom_importance.csv", index=False)
print("Symptoms that carry the largest share of the evidence when they are reported")
display(importance_df[["symptom", "mean influence (%)", "times reported"]])

# Phase 10: Confidence layer

Grades the top probability as VERY HIGH, HIGH, MODERATE or LOW. The table checks what those labels actually mean on test patients, so the app can show users how often each level was right.

In [ ]:
# =====================================================
# CONFIDENCE LEVELS VS REAL TEST ACCURACY
# =====================================================

top_probability = ensemble_test.max(axis=1)
top_correct = ensemble_test.argmax(axis=1) == y_test
confidence_names = [level for level, _ in config.CONFIDENCE_LEVELS] + [config.LOW_CONFIDENCE]
confidence_labels = np.select([top_probability >= t for _, t in config.CONFIDENCE_LEVELS],
                              confidence_names[:-1], default=config.LOW_CONFIDENCE)

CONFIDENCE_CALIBRATION = {}
for level in confidence_names:
    mask = confidence_labels == level
    CONFIDENCE_CALIBRATION[level] = {
        "n": int(mask.sum()),
        "share": float(mask.mean()),
        "accuracy": float(top_correct[mask].mean()) if mask.any() else 0.0,
    }

calibration_df = pd.DataFrame(CONFIDENCE_CALIBRATION).T
display(calibration_df.style.format({"n": "{:.0f}", "share": "{:.1%}", "accuracy": "{:.1%}"}))
print("Ideally accuracy falls as the confidence level falls.")

# Phase 11: Clinical assistant

Bug fixed: the old call `clinical_assistant(predicted_disease, confidence, level)` passed the **confidence** level as the **emergency** level, so the output read "Emergency Level : VERY HIGH". The emergency and confidence assessments are now separate objects.

In [ ]:
# =====================================================
# CLINICAL ASSISTANT
# =====================================================


def clinical_assistant(diagnosis):
    print("=" * 60)
    print("CLINICAL ASSISTANT")
    print("=" * 60)
    print(f"Most likely condition : {diagnosis.top.name}")
    print(f"Probability           : {diagnosis.top.probability:.2%}")
    print(f"Confidence level      : {diagnosis.confidence.level}")
    print(f"Emergency level       : {diagnosis.emergency.level} (score {diagnosis.emergency.score})")
    print("\nRecommendations:")
    for rec in diagnosis.recommendations:
        print(" -", rec)


clinical_assistant(result)
print()
urgent_result = predictor.diagnose([SYMPTOMS[i] for i in np.flatnonzero(X_test[example_row])])
clinical_assistant(urgent_result)

# Phase 12: Automated report generator

Creates the same PDF report that the web app offers for download.

In [ ]:
# =====================================================
# PDF REPORT
# =====================================================

pdf_bytes = build_pdf_report(result, {"name": "Sample test patient", "age": "", "sex": ""})
sample_pdf = config.REPORTS_DIR / "sample_clinical_report.pdf"
sample_pdf.write_bytes(pdf_bytes)
print(f"Saved {sample_pdf.relative_to(PROJECT_ROOT)} ({len(pdf_bytes) / 1024:.1f} KB)")

with open(config.REPORTS_DIR / "sample_result.json", "w", encoding="utf-8") as fh:
    json.dump(result.to_dict(), fh, indent=2)

# Phase 13: Evaluation and model saving

Per-disease metrics use disease names (the old report showed only numeric class ids). All artifacts are saved in formats that load without pickle problems:

| File | Content |
|---|---|
| `healthbandhu_dnn.keras` | Deep neural network |
| `healthbandhu_nb_params.npz` | Naive Bayes log probabilities, alpha, temperature |
| `healthbandhu_symptoms.json` | Input order of the symptoms |
| `healthbandhu_diseases.json` | Output order of the diseases |
| `healthbandhu_disease_profiles.npy` | Rule layer profiles |
| `model_metadata.json` | Measured metrics, ensemble weight, dataset facts, library versions |

In [ ]:
# =====================================================
# PER-DISEASE EVALUATION (ENSEMBLE)
# =====================================================

ensemble_pred = ensemble_test.argmax(axis=1)
present = np.unique(np.concatenate([y_test, ensemble_pred]))
report_dict = classification_report(y_test, ensemble_pred, labels=present,
                                    target_names=[DISEASES[i] for i in present],
                                    output_dict=True, zero_division=0)
per_disease = (pd.DataFrame(report_dict).T
               .drop(index=["accuracy", "macro avg", "weighted avg"], errors="ignore")
               .rename_axis("disease").reset_index())
per_disease["support"] = per_disease["support"].astype(int)
per_disease.to_csv(config.REPORTS_DIR / "per_disease_metrics.csv", index=False)

print("Hardest diseases with at least 20 test records")
display(per_disease[per_disease["support"] >= 20].sort_values("f1-score").head(15).round(3))

errors = pd.DataFrame({"true": np.array(DISEASES)[y_test], "predicted": np.array(DISEASES)[ensemble_pred]})
errors = errors[errors["true"] != errors["predicted"]]
confused = errors.value_counts().head(15).rename("count").reset_index()
confused.to_csv(config.REPORTS_DIR / "most_confused_pairs.csv", index=False)
print("Most frequently confused pairs")
display(confused)

In [ ]:
# =====================================================
# SAVE MODEL ARTIFACTS
# =====================================================


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


models_dir = config.MODELS_DIR
dnn_model.save(models_dir / config.DNN_MODEL_FILE)
np.savez_compressed(models_dir / config.NB_PARAMS_FILE, feature_log_prob=nb_flp, class_log_prior=class_log_prior,
                    temperature=np.float64(NB_TEMPERATURE), alpha=np.float64(NB_ALPHA))
np.save(models_dir / config.PROFILES_FILE, profiles.astype(np.float32))
with open(models_dir / config.SYMPTOMS_FILE, "w", encoding="utf-8") as fh:
    json.dump(SYMPTOMS, fh, indent=1)
with open(models_dir / config.DISEASES_FILE, "w", encoding="utf-8") as fh:
    json.dump(DISEASES, fh, indent=1)

metadata = {
    "app": config.APP_TITLE,
    "created_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "dataset": {
        "source_file": config.DATASET_FILENAME,
        "raw_rows": len(df_raw),
        "raw_symptom_columns": len(raw_symptom_cols),
        "raw_disease_classes": int(df_raw[TARGET].nunique()),
        "rows_used": int(len(y_train) + len(y_val) + len(y_test)),
        "symptoms_used": N_FEATURES,
        "diseases_used": N_CLASSES,
        "train_rows": int(len(y_train)),
        "val_rows": int(len(y_val)),
        "test_rows": int(len(y_test)),
        "min_samples_per_class": config.MIN_SAMPLES_PER_CLASS,
        "removed_rare_classes": REMOVED_CLASSES,
        "merged_duplicate_columns": DUPLICATE_COLUMNS,
        "dropped_zero_activation_symptoms": DROPPED_SYMPTOMS,
        "exact_duplicate_rows": exact_duplicate_rows,
        "records_in_conflicting_patterns": rows_in_conflict,
    },
    "naive_bayes": {"alpha": NB_ALPHA, "temperature": NB_TEMPERATURE},
    "ensemble": {"dnn_weight": DNN_WEIGHT, "nb_weight": 1.0 - DNN_WEIGHT,
                 "selected_by": "validation top-1 accuracy, ties broken by validation log loss"},
    "dnn": {"epochs_run": len(history.history["loss"]), "best_val_loss": float(min(history.history["val_loss"])),
            "parameters": int(dnn_model.count_params()), "hidden_layers": [512, 256, 128]},
    "rules": {"key_symptom_threshold": config.KEY_SYMPTOM_THRESHOLD, "key_symptoms_min": config.KEY_SYMPTOMS_MIN,
              "key_symptoms_max": config.KEY_SYMPTOMS_MAX, "f1_weight": config.RULE_F1_WEIGHT},
    "emergency": {"weights": config.EMERGENCY_SYMPTOMS, "levels": dict(config.EMERGENCY_LEVELS)},
    "metrics": {"test": RESULTS},
    "confidence_calibration": CONFIDENCE_CALIBRATION,
    "class_train_counts": class_train_counts,
    "symptom_train_frequency": train_frequency,
    "versions": VERSIONS,
}
with open(models_dir / config.METADATA_FILE, "w", encoding="utf-8") as fh:
    json.dump(to_builtin(metadata), fh, indent=2)

display(pd.DataFrame([{"file": p.name, "size (MB)": round(p.stat().st_size / 1024**2, 3)}
                      for p in sorted(models_dir.iterdir()) if p.is_file() and p.name != "README.md"]))

# Phase 14: Universal prediction pipeline

Reloads everything from disk exactly as the web app does, confirms the predictions are identical to the in-memory models, and runs the artifact checker script.

In [ ]:
# =====================================================
# RELOAD FROM DISK AND VERIFY
# =====================================================

loaded_predictor = HealthBandhuPredictor(load_bundle(config.MODELS_DIR))
check = X_test[:1024].astype("float32")
difference = float(np.abs(loaded_predictor.ensemble_proba(check) - predictor.ensemble_proba(check)).max())
print(f"Max difference after reload: {difference:.2e}")
assert difference < 1e-5, "Reloaded models do not reproduce the trained models"
print("Loading notes:", loaded_predictor.bundle.warnings or "none")


def predict_disease(symptoms, top_k=config.TOP_K):
    """Universal prediction entry point: list of symptom names -> DiagnosisResult."""
    return loaded_predictor.diagnose(symptoms, top_k=top_k)


demo_symptoms = [s for s in ["cough", "fever", "sore throat", "nasal congestion", "headache"] if s in SYMPTOMS]
demo = predict_disease(demo_symptoms)
print("\nSymptoms  :", demo.symptoms)
print("Emergency :", demo.emergency.level)
print("Top 5     :", [(c.name, round(c.probability, 3)) for c in demo.candidates])

checker = subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "verify_artifacts.py")],
                         capture_output=True, text=True, cwd=PROJECT_ROOT)
print(checker.stdout[-3000:])
if checker.returncode != 0:
    print(checker.stderr[-3000:])
    raise RuntimeError("Artifact verification failed")

# Phase 15: Packaging and launch

- **Release zip:** bundles the project with the trained models (the dataset is excluded) into `HEALTHBANDHU V.zip` next to the project folder. In Colab it downloads automatically.
- **Run locally:** extract the zip, then `pip install -r requirements.txt` and `streamlit run App.py`.
- **Public link from Colab:** set `LAUNCH_APP_IN_COLAB = True` and run the last cell. It starts Streamlit, waits until it is healthy, then opens a Cloudflare tunnel. (The old launcher started `app.py` before that file existed.)

In [ ]:
# =====================================================
# CREATE RELEASE ZIP (CODE + TRAINED MODELS)
# =====================================================

CREATE_RELEASE_ZIP = True

EXCLUDED_DIRS = {"__pycache__", ".ipynb_checkpoints", ".git", "tools", ".venv", "venv"}
release_zip = PROJECT_ROOT.parent / f"{PROJECT_NAME}.zip"

if CREATE_RELEASE_ZIP:
    added = 0
    with zipfile.ZipFile(release_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(PROJECT_ROOT.rglob("*")):
            rel = path.relative_to(PROJECT_ROOT)
            if path.is_dir() or EXCLUDED_DIRS.intersection(rel.parts) or path.suffix == ".pyc":
                continue
            if rel.parts[0] == "data" and path.name != "README.md":
                continue  # the 190 MB dataset downloads automatically
            zf.write(path, Path(PROJECT_NAME) / rel)
            added += 1
    print(f"Created {release_zip} with {added} files ({release_zip.stat().st_size / 1024**2:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(str(release_zip))

In [ ]:
# =====================================================
# OPTIONAL: LAUNCH THE WEB APP FROM COLAB WITH A PUBLIC LINK
# =====================================================

LAUNCH_APP_IN_COLAB = False

if LAUNCH_APP_IN_COLAB:
    import runpy

    launcher = runpy.run_path(str(PROJECT_ROOT / "scripts" / "colab_launch.py"))
    launcher["launch"](port=8501, tunnel=True)
else:
    print("Launch skipped. Set LAUNCH_APP_IN_COLAB = True to start the app with a public link.")